# P96 — Un nuevo enfoque para los problemas de filtrado y predicción lineales

## 1. Título y paper

**Paper:** *A New Approach to Linear Filtering and Prediction Problems*  
**Autoría:** Rudolf E. Kálmán  
**Año y venue:** 1960 · Journal of Basic Engineering, 82(1), 35–45  
**Nivel:** L3 · **Motor:** `kalman`  
**Ficha completa:** [`P96_kalman`](../../papers/foundational/P96_kalman/README.md)

**Hito:** Fusiona un modelo del movimiento con un sensor ruidoso ponderando cada fuente por su propia incertidumbre, y lo hace de forma recursiva.

- [doi:10.1115/1.3662552](https://doi.org/10.1115/1.3662552)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Un sensor da medidas ruidosas y un modelo del movimiento acumula error. Promediarlos trata igual a los dos, e ignora que la confianza en cada uno cambia con el tiempo. Los métodos anteriores exigían guardar todo el historial.
2. Ejecutar una implementación mínima de la propuesta: Mantener una estimación y su varianza, predecir con el modelo, y corregir con la medida usando una ganancia que sale del cociente entre las dos incertidumbres. Recursivo: solo hace falta el estado anterior.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- Wiener (1949), filtrado óptimo en el dominio de la frecuencia
- P87


## 4. Intuición

Tienes un modelo que dice dónde deberías estar y un sensor que dice dónde pareces estar. Los dos mienten un poco. La respuesta no es promediarlos: es hacer más caso al que en este momento sea menos incierto — y ese peso cambia solo, paso a paso.


## 5. Concepto mínimo

```text
Predecir:   x̂ ← x + u            P ← P + Q
Corregir:   K ← P / (P + R)        ← la GANANCIA sale de las dos varianzas
            x ← x̂ + K·(z − x̂)      P ← (1 − K)·P

    Q = incertidumbre del modelo      R = incertidumbre del sensor
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('kalman', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuál será el error del sensor solo? ¿Y del modelo solo?
2. ¿Y el del filtro, que no usa ninguna información adicional?
3. ¿Qué le pasa a la ganancia con el tiempo?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('kalman', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('kalman', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

El sensor solo da 1,8895 de error y el modelo sin corregir, 0,7943. El filtro —**las mismas dos fuentes**— baja a 0,4559. Y la ganancia arranca en 0,208 y baja a 0,106: al principio la estimación es mala y se hace caso al sensor; después el modelo ya sabe dónde está.


## 10. Comentario pedagógico

Lo importante es que nadie ajusta esa ponderación. Sale del cociente entre las varianzas, y por eso con un sensor diez veces peor la ganancia cae sola a 0,034. Es un mecanismo de **desconfianza automática**, y es la razón de que el mismo filtro sirva para un cohete y para un GPS de móvil.


## 11. Error o anti-patrón deliberado

Anti-patrón: usar una media móvil donde hace falta un filtro.


In [ ]:
print('Una media movil trata todas las medidas por igual e ignora que hay un modelo.')
print('Ademas introduce retraso: la estimacion va siempre por detras de la realidad.')
print('En la miniatura da 2.2179 de error frente a 0.4559 del filtro.')

## 12. Corrección

La diferencia, con las cuatro estrategias sobre los mismos datos:


In [ ]:
r = run_paper_lab('kalman', seed=7)['result']
for k in ('error_solo_sensor', 'error_solo_modelo', 'error_media_movil_5', 'error_filtro_de_kalman'):
    print(f'{k:<28} {r[k]}')
print()
print('ganancia inicial / final :', r['ganancia_inicial'], '/', r['ganancia_final'])
print('con sensor 10x peor      :', r['ganancia_con_sensor_10x_peor'])

## 13. Desafío guiado

Sigue la ganancia paso a paso y explica por qué baja aunque el sensor no cambie.


In [ ]:
r = run_paper_lab('kalman', seed=3)['result']
show(r)

## 14. Desafío autónomo

Implementa un filtro de Kalman para una señal real de tu trabajo —temperatura, latencia, posición— estimando Q y R de los datos. Compara con una media móvil y documenta el retraso de cada una.


## 15. Evidencia de aprendizaje

Guarda la comparación de los cuatro estimadores y tu explicación de de dónde sale la ganancia.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P96_kalman/README.md) · evaluación formal: [`assessments/papers/P96_kalman.md`](../../assessments/papers/P96_kalman.md)


## 16. Cierre

Ya se sabe dónde está el robot. La pregunta siguiente es qué arquitectura lo hace actuar: ¿modelo del mundo y plan, o reflejos?


## 17. Conexión con el siguiente hito

- P99

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
